# Favorita completo: entrenamiento y recuperacion en Colab
Ejecutar las celdas en orden. Primero publicar este bloque en el repositorio; el notebook no hace push.
Los CSV permanecen en Drive y se copian a /content para lectura local. Se usa el MLP y entrenamiento existentes, sin evaluar test.
Cada epoca completada se guarda localmente y se copia a una **nueva generacion verificada** en Drive.
No usar la carpeta ni checkpoints del piloto de dos tiendas: las dimensiones y categorias son diferentes.


In [ ]:
# 1. Parametros explicitos. No incluir tokens en la URL ni compartir rutas personales.
REPO_URL = "https://github.com/OverCaivano/Redes-Neuronales.git"
REVISION = "main"  # Para reanudar, usar el commit mostrado en la primera sesion.
TRAIN_CSV_DRIVE = ""  # Completar con la ruta absoluta bajo /content/drive/.
RESULTADOS_DRIVE = ""  # Carpeta EXCLUSIVA para esta corrida completa.
MODO = "nuevo"  # "nuevo" o "reanudar"
EPOCAS_TOTALES = 10
BATCH_SIZE = 256
DISPOSITIVO = "cpu"  # "cpu" o "cuda"; elegir GPU en Entorno de ejecucion si corresponde.
SEMILLA = 42
EJECUTAR_ENTRENAMIENTO = False  # Cambiar a True cuando se hayan revisado los parametros.


## Codigo y entorno aislado
Se clona la URL configurada. Se crea un venv separado del kernel de Colab para evitar que paquetes ya importados usen otras versiones.
Las versiones fijadas corresponden al entorno observado del proyecto (PyTorch 2.13.0 y sklearn 1.9.0).
Si pip no encuentra ruedas compatibles con el Python/runtime disponible, **detenerse**: no quitar pins para reanudar.
No se instalan torchvision ni pytest: no son necesarios para este entrenamiento.


In [ ]:
# 2. Obtener exactamente la revision solicitada, sin sobrescribir un checkout existente.
import json, os, subprocess, sys, uuid
from pathlib import Path
REPO = Path("/content") / ("favorita_codigo_" + uuid.uuid4().hex)
if not REPO_URL.startswith("https://github.com/") or "@" in REPO_URL:
    raise ValueError("Configurar una URL HTTPS de GitHub sin credenciales")
subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "checkout", "--detach", REVISION], cwd=REPO, check=True)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
print("Guardar esta revision para reanudar:", COMMIT)
for nombre in ["requirements-colab.txt", "src/sincronizacion.py", "configs/favorita_completa.yaml"]:
    if not (REPO / nombre).is_file():
        raise FileNotFoundError("Publicar primero el Bloque 1E-C: " + nombre)
os.chdir(REPO)


In [ ]:
# 3. Entorno dedicado: no sustituye los paquetes del kernel ni requiere reiniciarlo.
ENTORNO = REPO / ".venv-colab"
subprocess.run([sys.executable, "-m", "venv", str(ENTORNO)], check=True)
PYTHON = str(ENTORNO / "bin" / "python")
subprocess.run([PYTHON, "-m", "pip", "install", "-r", "requirements-colab.txt"], check=True)
subprocess.run([PYTHON, "-m", "pip", "check"], check=True)
subprocess.run([PYTHON, "-c", "from src.sincronizacion import entorno_actual; print(entorno_actual())"], check=True)


## Drive y datos
Subir `train.csv` a Drive por separado, nunca al repositorio. Completar las dos rutas en la primera celda.
El montaje requiere autorizar la cuenta de Google. El CSV se verifica con SHA-256 al copiarse; no se modifica el original.
Mantener espacio para el CSV local, CSR/metadatos y varias generaciones de checkpoints en Drive.


In [ ]:
# 4. Montar Drive y validar destinos antes de escribir.
from google.colab import drive
drive.mount("/content/drive")
if MODO not in ("nuevo", "reanudar") or DISPOSITIVO not in ("cpu", "cuda"):
    raise ValueError("MODO o DISPOSITIVO invalido")
if EPOCAS_TOTALES < 1 or BATCH_SIZE < 1:
    raise ValueError("Epocas y batch_size deben ser positivos")
if not TRAIN_CSV_DRIVE or not RESULTADOS_DRIVE:
    raise ValueError("Completar TRAIN_CSV_DRIVE y RESULTADOS_DRIVE en la primera celda")
for valor in (TRAIN_CSV_DRIVE, RESULTADOS_DRIVE):
    if Path("/content/drive") not in Path(valor).resolve().parents:
        raise ValueError("Las rutas persistentes deben estar bajo el montaje de Drive")
if not Path(TRAIN_CSV_DRIVE).is_file():
    raise FileNotFoundError(TRAIN_CSV_DRIVE)
if MODO == "nuevo" and Path(RESULTADOS_DRIVE).exists() and any(Path(RESULTADOS_DRIVE).iterdir()):
    raise ValueError("Corrida nueva requiere carpeta Drive vacia; usar otra o MODO=reanudar")
CSV_LOCAL = Path("/content/favorita_datos/train.csv")
RUN_LOCAL = Path("/content/favorita_runs") / uuid.uuid4().hex
CONFIG_LOCAL = Path("/content") / ("favorita_config_" + uuid.uuid4().hex + ".yaml")


## Preparar una corrida nueva o recuperar la ultima epoca integra
`nuevo` usa todas las tiendas y familias de `configs/favorita_completa.yaml`, con input_dim automatico.
`reanudar` restaura los cinco archivos a una carpeta local nueva. Verifica hashes, entorno, codigo, configuracion y CSV antes de entrenar.
Mantener semilla, batch y dispositivo originales al reanudar; solo cambian rutas y numero **total** de epocas.
No se garantiza igualdad bit a bit entre hardware/CPU/GPU. CUDA disponible no implica que este MLP pequeno sea mas rapido: comparar tiempos por epoca en corridas independientes.


In [ ]:
# 5. Preparacion en el mismo interprete aislado que entrenara.
parametros = dict(modo=MODO, csv_drive=TRAIN_CSV_DRIVE, remoto=RESULTADOS_DRIVE,
                  csv_local=str(CSV_LOCAL), local=str(RUN_LOCAL), config=str(CONFIG_LOCAL),
                  epochs=EPOCAS_TOTALES, batch=BATCH_SIZE, device=DISPOSITIVO, seed=SEMILLA)
preparar = r"""
import copy, json, sys
from pathlib import Path
import torch, yaml
from src.sincronizacion import copiar_csv_local, restaurar_ultima_copia
from src.reanudacion import cargar_last, validar_reanudacion
p = json.load(sys.stdin)
print('CUDA disponible:', torch.cuda.is_available(), flush=True)
if p['device'] == 'cuda' and not torch.cuda.is_available():
    raise ValueError('CUDA solicitada pero no disponible; revisar runtime')
digest = copiar_csv_local(p['csv_drive'], p['csv_local'])
print('SHA-256 train.csv:', digest, flush=True)
if p['modo'] == 'reanudar':
    last = restaurar_ultima_copia(p['remoto'], p['local'])
    estado = cargar_last(last)
    config = copy.deepcopy(estado['config'])
    if config['data'].get('tiendas') is not None or config['data'].get('familias') is not None:
        raise ValueError('No se permiten checkpoints del subconjunto')
    print('Epoca recuperada:', estado['epoch'], 'mejor epoca:', estado['best_epoch'], flush=True)
else:
    config = yaml.safe_load(Path('configs/favorita_completa.yaml').read_text())
config['data']['path'] = p['csv_local']
config['output']['directory'] = p['local']
config['training'].update(epochs=p['epochs'], batch_size=p['batch'])
config['device'], config['seed'] = p['device'], p['seed']
if p['modo'] == 'reanudar':
    resumen = validar_reanudacion(last, estado, config, digest)
    print('Ejemplos:', resumen['filas_por_particion'], flush=True)
Path(p['config']).write_text(yaml.safe_dump(config, sort_keys=False))
print('Dispositivo:', p['device'], 'resultados locales:', p['local'], flush=True)
print('Resultados persistentes:', p['remoto'], flush=True)
"""
subprocess.run([PYTHON, "-u", "-c", preparar], input=json.dumps(parametros), text=True, check=True)


## Entrenar con el pipeline existente
Esta celda inicia la corrida completa **solo si** se habilito EJECUTAR_ENTRENAMIENTO. No se entrena al preparar el notebook.
Cada epoca imprime ejemplos, MAE/RMSE, mejor epoca y destino verificado. Test no se evalua.
`--sync-dir` espera a verificar la copia de cada epoca antes de continuar. Un error de Drive detiene la corrida y conserva los archivos locales y generaciones previas.


In [ ]:
# 6. Sin logica de entrenamiento duplicada. Total de epocas, no epocas adicionales.
comando = [PYTHON, "-u", "-m", "src.train", "--config", str(CONFIG_LOCAL),
           "--epochs", str(EPOCAS_TOTALES), "--sync-dir", RESULTADOS_DRIVE]
if MODO == "reanudar":
    comando += ["--resume", str(RUN_LOCAL / "last.pt")]
print("Comando:", comando)
if EJECUTAR_ENTRENAMIENTO:
    subprocess.run(comando, check=True)
else:
    print("Entrenamiento deshabilitado. Revisar parametros y habilitarlo explicitamente.")


In [ ]:
# 7. Ver resultados locales sin cargar el dataset ni evaluar test.
import csv
if (RUN_LOCAL / "resumen.json").exists():
    resumen = json.loads((RUN_LOCAL / "resumen.json").read_text())
    print(json.dumps(resumen, indent=2))
    with (RUN_LOCAL / "metricas.csv").open(newline="") as archivo:
        for fila in csv.DictReader(archivo):
            print({k: fila[k] for k in ("epoca", "val_mae", "val_rmse", "baseline_rmse")})
    print("Copias persistentes:", RESULTADOS_DRIVE)
else:
    print("Todavia no hay epocas completadas en esta sesion.")


## Recuperacion despues de una desconexion
1. Abrir nuevamente este notebook y seleccionar la misma revision/commit, versiones y dispositivo.
2. Usar las mismas rutas de Drive, `MODO = "reanudar"` y el total deseado de epocas.
3. Ejecutar las celdas en orden. Se ignoran con advertencia las generaciones incompletas o corruptas; se restaura la ultima integra.
4. Se reutilizan pesos, Adam, transformaciones, historial y RNG. No se reutiliza el checkpoint del piloto. Si ya se alcanzo el total, no se ejecutan epocas nuevas.

Si falla solo la sincronizacion y la VM sigue viva, se puede reintentar **sin entrenar** desde una celda:
```python
subprocess.run([PYTHON, "-c", "import sys; from src.sincronizacion import sincronizar_epoca; print(sincronizar_epoca(sys.argv[1], sys.argv[2]))", str(RUN_LOCAL), RESULTADOS_DRIVE], check=True)
```
Luego recuperar normalmente con `MODO="reanudar"`. No ejecutar dos sesiones escribiendo en la misma carpeta.
Cada generacion tiene cinco artefactos, manifiesto SHA-256 y marca COMPLETO. No se borran copias automaticamente.
La verificacion por lectura detecta copias parciales/corrupcion; el montaje de Drive no ofrece una transaccion distribuida ni garantiza persistencia remota inmediata. Si se pierde la VM durante una epoca o transferencia, se vuelve a la ultima generacion integra disponible.
Los checkpoints contienen objetos Python: solo restaurar archivos propios y confiables. Los hashes no autentican su procedencia.
RAM y GPU dependen del runtime; el dataset permanece CSR y solo se densifican lotes. La disponibilidad de ruedas de las versiones fijadas y CUDA se comprueba en Colab, no se supone.
Referencias: [Drive/limites Colab](https://research.google.com/colaboratory/faq.html), [persistencia sklearn](https://scikit-learn.org/stable/model_persistence.html).
